# Phase 4 — Ranking + End-to-End Evaluation

Loads artifacts produced by phases 1–3 and runs:
1. LightGBM LambdaRank on `(customer, candidate)` pairs using SASRec retrieval + churn score + customer/item features.
2. Recsys metrics (Recall@K, NDCG@K) on the test window.
3. Business slice: performance on the high-churn segment + retention hit-rate.
4. Ablation: retrieval-only vs +ranker vs +ranker+LLM.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data.splits import build_windows
from src.features.build_features import build_customer_features
from src.models.ranking.lgbm_ranker import build_pairs, train_ranker, rerank
from src.evaluation.recsys_metrics import evaluate_recsys
from src.evaluation.business import evaluate_by_risk_segment, retention_impact

PROC = PROJECT_ROOT / 'data' / 'processed'
FEAT = PROJECT_ROOT / 'data' / 'features'
df = pd.read_parquet(PROC / 'transactions_clean.parquet')
windows = build_windows(df, horizon_days=90, n_windows=3)
labels = {n: pd.read_parquet(PROC / f'churn_labels_{n}.parquet') for n in ['train','val','test']}
candidates = pd.read_parquet(FEAT / 'retrieval_candidates_test.parquet')
churn_gbm = pd.read_parquet(FEAT / 'churn_scores_gbm_test.parquet')
candidates.head()

## Build ground truth for the test label window

In [ ]:
w_test = windows[2]
label_window_purchases = df[(df['invoice_date'] >= w_test.feature_end) & (df['invoice_date'] < w_test.label_end)]
print(f"Purchases in test label window: {len(label_window_purchases):,} rows, {label_window_purchases['customer_id'].nunique():,} customers")

## Train LightGBM ranker on the train window

We need training candidates with relevance labels. We build them by:
1. Taking train-window retrieval candidates (need to run SASRec/FAISS for train window — for brevity here we re-use test-window candidate structure; in production this is a separate offline build step)
2. Marking items the customer actually purchased in the train *label* window as relevant.

In [ ]:
# Customer features at test.feature_end (we use the same set used by the GBM churn model)
top_countries = df['country'].value_counts().head(8).index.tolist()
cust_feats = build_customer_features(df, w_test.feature_end, country_dummies=top_countries)

# Item features — popularity, avg price, recency
pre = df[df['invoice_date'] < w_test.feature_end]
item_feats = pre.groupby('stock_code').agg(
    item_n_orders=('invoice','nunique'),
    item_n_customers=('customer_id','nunique'),
    item_avg_price=('price','mean'),
    item_total_revenue=('revenue','sum'),
).reset_index()
item_feats['item_log_orders'] = np.log1p(item_feats['item_n_orders'])
print(item_feats.shape, cust_feats.shape)

In [ ]:
churn_for_join = churn_gbm.rename(columns={'p_churn_gbm': 'p_churn'})

pairs = build_pairs(
    candidates=candidates,
    interactions_in_label_window=label_window_purchases,
    customer_features=cust_feats,
    item_features=item_feats,
    churn_scores=churn_for_join,
)
print(f"Pairs: {len(pairs):,}  |  relevance=1 rate: {pairs['relevance'].mean()*100:.3f}%")

In [ ]:
feature_cols = [
    'score','log_score','inv_rank','rank',
    'p_churn',
    'recency_days','frequency','monetary','mean_interval_days','product_diversity',
    'item_log_orders','item_n_customers','item_avg_price',
]
feature_cols = [c for c in feature_cols if c in pairs.columns]

# Split pairs by customer for an honest holdout: 80/20 of customers
cust_ids = pairs['customer_id'].drop_duplicates().sample(frac=1, random_state=42).tolist()
n_tr = int(0.8 * len(cust_ids))
tr_cust = set(cust_ids[:n_tr])

tr = pairs[pairs['customer_id'].isin(tr_cust)]
te = pairs[~pairs['customer_id'].isin(tr_cust)]

model = train_ranker(tr, feature_cols)
reranked = rerank(model, te, feature_cols, k=10)
print(f"Top-10 reranked for {reranked['customer_id'].nunique():,} customers")

## Recsys metrics — Recall@K, NDCG@K

In [ ]:
gt = label_window_purchases[['customer_id','stock_code']]

# Ablation: retrieval-only (top-K by FAISS score) vs +ranker
retrieval_only = (
    candidates[candidates['customer_id'].isin(reranked['customer_id'].unique())]
    .sort_values(['customer_id','rank']).groupby('customer_id').head(10)
)

metrics_retrieval = evaluate_recsys(retrieval_only[['customer_id','stock_code']], gt)
metrics_ranker = evaluate_recsys(reranked[['customer_id','stock_code']], gt)

ablation = pd.DataFrame({'retrieval_only': metrics_retrieval, 'retrieval+ranker': metrics_ranker})
ablation

## Business slices — high-churn segment + retention impact

In [ ]:
yte = labels['test'].merge(churn_gbm, on='customer_id', how='inner')
by_seg = evaluate_by_risk_segment(yte['customer_id'].values, yte['churn'].values, yte['p_churn_gbm'].values)
by_seg

In [ ]:
high_risk = yte[yte['p_churn_gbm'] >= yte['p_churn_gbm'].quantile(0.8)]
churned = set(labels['test'][labels['test']['churn']==1]['customer_id'].astype(int))
recs_map = reranked.sort_values(['customer_id','final_rank']).groupby('customer_id')['stock_code'].apply(list).to_dict()
impact = retention_impact(high_risk[['customer_id']], churned, recs_map, label_window_purchases)
impact

In [ ]:
# Persist final ablation table
ablation.to_csv(PROJECT_ROOT / 'reports' / 'recsys_ablation.csv')
by_seg.to_csv(PROJECT_ROOT / 'reports' / 'churn_by_risk_segment.csv', index=False)
print('Saved reports/recsys_ablation.csv and reports/churn_by_risk_segment.csv')